# P1_04 · U-Net baseline on Kvasir-SEG (P1-10)

Trains `smp.Unet(resnet34, imagenet)` on the Kvasir train split (700 images, 352×352, 60 epochs, AdamW, cosine schedule, BCE + Dice), keeps the checkpoint with the best val Dice, and evaluates it once on the same 200 test images as the foundation models. Roughly 20–30 min on a T4.

**Resumable**: the last checkpoint is saved to Drive after every epoch; after a disconnect just *Run all* again.

Outputs: `MyDrive/promptshift/results/raw/<EXP_ID>/unet.csv`, `unet_history.csv`, `unet_train.json`; checkpoints in `MyDrive/promptshift/checkpoints/<EXP_ID>/`. At the end a zip of the result files is downloaded: unzip it into the repo folder `results/` and tell Claude Code. Needs P1_00 first.

**Before you start** (once per Google account):
1. Colab left bar → 🔑 *Secrets* → add `GH_TOKEN` (a GitHub fine-grained token with read access
   to `geekRoy0126/promptshift`) and switch on *Notebook access*. Never paste the token into a cell.
2. *Runtime → Change runtime type → T4 GPU*.
3. *Runtime → Run all*. Allow Google Drive access when asked.

All data, weights and results go to `MyDrive/promptshift/`, so a disconnect loses nothing:
rerun the notebook and finished steps are skipped.

In [ ]:
# Parameters (edit before Run all if needed)
EXP_ID = "p1_kvasir_pilot"
EPOCHS = None  # None = config value (60); a small number = quick smoke run

In [ ]:
# SETUP: identical in every notebook (generated by scripts/build_notebooks.py).
import base64
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/geekRoy0126/promptshift.git"
REF = "main"  # branch, tag or commit to run

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    REPO = Path("/content/promptshift")
    try:
        _token = userdata.get("GH_TOKEN")
    except Exception as e:  # missing secret or notebook access not granted
        raise RuntimeError(
            "Add the Colab secret GH_TOKEN (key icon in the left bar) and switch on "
            "'Notebook access' for it."
        ) from e
    _auth = base64.b64encode(f"x-access-token:{_token}".encode()).decode()

    def _git(*args, auth=False):
        pre = ["-c", f"http.extraHeader=Authorization: Basic {_auth}"] if auth else []
        r = subprocess.run(["git", *pre, "-C", str(REPO), *args], capture_output=True, text=True)
        if r.returncode:
            msg = (r.stderr or r.stdout).replace(_token, "***").replace(_auth, "***")
            raise RuntimeError(f"git {args[0]} failed: {msg}")

    REPO.mkdir(parents=True, exist_ok=True)
    if not (REPO / ".git").is_dir():
        _git("init", "-q")
    _git("fetch", "-q", "--depth", "1", REPO_URL, REF, auth=True)
    _git("reset", "-q", "--hard", "FETCH_HEAD")  # throwaway clone; all outputs live on Drive
    del _token, _auth
    _pip = [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)]
    r = subprocess.run(_pip, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("pip install -e failed (log above)")
else:  # local CPU run: use the checkout this notebook lives in
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
os.environ["PYTHONUNBUFFERED"] = "1"
# scripts must import this checkout's code, whatever else is installed
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO / "src"), os.environ.get("PYTHONPATH", "")])


def run(*args, check=True):
    """Run a repo script with live output; stop the notebook if it fails. Returns the output."""
    cmd = [sys.executable, *map(str, args)]
    print("$ python", " ".join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = []
    while chunk := p.stdout.read1(4096):
        text = chunk.decode(errors="replace")
        out.append(text)
        sys.stdout.write(text)
        sys.stdout.flush()
    if p.wait() and check:
        raise RuntimeError(f"exit code {p.returncode}: python {' '.join(cmd[1:])}")
    return "".join(out)


def check_gpu():
    import torch

    if IN_COLAB and not torch.cuda.is_available():
        raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then Run all again.")
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")


def git_out(*args):
    r = subprocess.run(["git", "-C", str(REPO), *args], capture_output=True, text=True)
    return r.stdout.strip()


SHA = git_out("rev-parse", "--short", "HEAD")
print(f"repo {REPO} @ {SHA} (Colab: {IN_COLAB})")

In [ ]:
check_gpu()

## 1. Train and test

In [ ]:
args = ["scripts/train_unet.py", "--exp-id", EXP_ID]
if EPOCHS:
    args += ["--epochs", EPOCHS]
run(*args)

## 2. Result

In [ ]:
import json

import pandas as pd
from IPython.display import display

from promptshift.paths import get_paths

paths = get_paths()
exp_dir = paths.raw_results / EXP_ID
record = json.loads((exp_dir / "unet_train.json").read_text())
print(json.dumps(record, indent=1))
display(pd.read_csv(exp_dir / "unet_history.csv").tail())
test = pd.read_csv(exp_dir / "unet.csv")
assert len(test) == record["n_test"] and (test["flag"] == "ok").all()

## 3. Export

In [ ]:
import zipfile

export = paths.root / "export"
export.mkdir(exist_ok=True)
zip_path = export / f"p1_04_{EXP_ID}_{SHA}.zip"
parts = sorted(exp_dir.glob("unet*"))
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for part in parts:
        files = [part] if part.is_file() else sorted(part.rglob("*"))
        for f in files:
            if f.is_file():
                z.write(f, f.relative_to(paths.results))
print(f"wrote {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MB)")
if IN_COLAB:
    from google.colab import files

    files.download(str(zip_path))
print("\nP1_04 done. Unzip it into the repo folder results/ and tell Claude.")